# AI Response Evaluation Environment — GRPO RL Training

**Meta PyTorch OpenEnv Hackathon**

Trains `Qwen2.5-1.5B-Instruct` using GRPO against the `AIResponseEvalEnvironment`.
The environment is used **directly in Python** (no HTTP server).

| Step | What happens |
|---|---|
| 1 | Clone environment repo + install deps |
| 2 | Configure model + HF token |
| 3 | Import environment + smoke test |
| 4 | Define TRL tool environment wrapper |
| 5 | Load Qwen2.5-1.5B with LoRA via Unsloth |
| 6 | GRPO training (300 steps, ~30 min on T4) |
| 7 | Plot reward curves |
| 8 | Save LoRA + merged model |
| 9 | Push to HuggingFace Hub |
| 10 | Before vs After comparison |

**Runtime:** T4 GPU (free Colab tier) or HF Spaces GPU (~$0.30 credit)


## Cell 1 — Install dependencies

In [1]:
# Install training stack
!pip install unsloth trl transformers datasets peft accelerate bitsandbytes -q
!pip install 'openenv-core[core]>=0.2.1' matplotlib -q

# Clone the eval environment from HF Spaces
import os, sys
!git clone https://huggingface.co/spaces/rsaibhargav/ai-response-eval-env /content/ai_response_eval_env -q 2>/dev/null || echo 'Already cloned'
!pip install -r /content/ai_response_eval_env/requirements.txt -q
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/ai_response_eval_env')
os.chdir('/content/ai_response_eval_env')
print('Done')


## Cell 2 — Configuration

In [ ]:
import os

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_STEPS  = 300
NUM_GEN    = 4
LORA_RANK  = 16

# Set HF_TOKEN via: Runtime → Secrets → Add secret → HF_TOKEN
HF_TOKEN = os.getenv('HF_TOKEN', '')
HF_REPO  = 'rsaibhargav/ai-response-eval-grpo'

print(f'Model : {MODEL_NAME}')
print(f'Steps : {MAX_STEPS}')
print(f'Token : {"set" if HF_TOKEN else "NOT SET — push will be skipped"}')


## Cell 3 — Import environment + build grading dataset


In [ ]:
import sys, os, requests, random
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/ai_response_eval_env')
os.chdir('/content/ai_response_eval_env')
from server.ai_response_eval_environment import AIResponseEvalEnvironment, PROBLEMS, TASK_TYPES
print('Environment imported OK — tasks:', list(TASK_TYPES.values()))
print(f'Problems loaded: {sum(len(v) for v in PROBLEMS.values())} total across {len(PROBLEMS)} difficulties')

## Cell 4 — Build dataset and reward function


In [ ]:
from datasets import Dataset

SPACE_URL = 'https://rsaibhargav-ai-response-eval-env.hf.space'

DIFFICULTY_MAP = {
    'correctness_check':      'easy',
    'tone_appropriateness':   'medium',
    'multi_dimensional':      'hard',
    'conversation_coherence': 'ultra',
    'adversarial_check':      'adversarial',
}

SYSTEM_PROMPT = (
    'You are an expert AI response evaluator. '
    'Read the scenario and reply with ONLY the verdict in the exact format shown. '
    'No explanation, no extra words — just the verdict string.\n\n'
    'EXACT formats:\n'
    'correctness_check → "correct, none" | "incorrect, factual-error" | '
    '"incorrect, instruction-violation" | "incorrect, fabricated" | "partially-correct, incomplete"\n'
    'tone_appropriateness → "appropriate, none" | "inappropriate, insensitive" | '
    '"inappropriate, too-technical" | "inappropriate, tone-mismatch" | '
    '"inappropriate, age-inappropriate" | "inappropriate, harmful" | "inappropriate, biased"\n'
    'multi_dimensional → "correctness=N, tone=N, empathy=N, safety=N"  (N = 0-10)\n'
    'conversation_coherence → "consistent=yes, contradictions=0, context_loss=no" | '
    '"consistent=no, contradictions=2, context_loss=yes"\n'
    'adversarial_check → "issue=injection, severity=high" | "issue=injection, severity=medium" | '
    '"issue=format_violation, severity=low" | "issue=none, severity=none"'
)

# Build dataset: each row = one real problem scenario + grading metadata
rows = []
for task_type, difficulty in DIFFICULTY_MAP.items():
    probs = PROBLEMS.get(difficulty, [])
    for i, prob in enumerate(probs):
        scenario = prob.get('scenario', prob.get('test_case_input', str(prob)))
        rows.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': f'TASK: {task_type}\n\nSCENARIO:\n{scenario}\n\nYour verdict:'},
            ],
            'task_type':     task_type,
            'difficulty':    difficulty,
            'problem_index': i,
        })

random.seed(42)
random.shuffle(rows)
dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} problems ({len(DIFFICULTY_MAP)} tasks)')
print('Columns:', dataset.column_names)

# ── Reward function — grades via live /grader endpoint (stateless, exact) ─────
def compute_reward(completions, task_type, difficulty, problem_index, **kwargs):
    """
    Standard TRL reward_funcs signature:
      completions   — list[str], one per generation in the GRPO batch
      task_type     — list[str], from dataset column
      problem_index — list[int], from dataset column
    Returns list[float] scores in [0, 1].
    """
    scores = []
    for completion, task, idx in zip(completions, task_type, problem_index):
        answer = completion.strip().strip('\"\' ')
        try:
            r = requests.post(
                f'{SPACE_URL}/grader',
                json={'task_id': task, 'answer': answer, 'problem_index': int(idx)},
                timeout=20,
            )
            score = float(r.json().get('score', 0.0)) if r.ok else 0.0
        except Exception as e:
            print(f'  [grader error] {e}')
            score = 0.0
        scores.append(score)
    return scores

# Smoke test
_row = dataset[0]
_s = compute_reward(
    completions=['correct, none'],
    task_type=[_row['task_type']],
    difficulty=[_row['difficulty']],
    problem_index=[_row['problem_index']],
)
print(f'Reward smoke test → task={_row["task_type"]}  score={_s[0]:.2f}  (endpoint reachable)')

## Cell 5 — Load model with Unsloth + LoRA

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 700

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,         # auto-detect bfloat16 on T4
    load_in_4bit=True,  # 4-bit quantisation — halves VRAM
)

# Apply LoRA — ONLY these adapter weights will be trained
# Everything else (base model) stays frozen
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print('LoRA adapters applied — only adapter weights will update during training')

## Cell 6 — GRPO Training


In [ ]:
from trl import GRPOConfig, GRPOTrainer

reward_log = []

class RewardLogCallback:
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'reward' in logs:
            entry = {'step': state.global_step, 'reward': logs['reward']}
            reward_log.append(entry)
            print(f'Step {state.global_step:4d} | reward={logs["reward"]:.4f}')

training_args = GRPOConfig(
    num_generations=NUM_GEN,
    max_prompt_length=512,
    max_completion_length=50,   # verdicts are short strings
    temperature=0.9,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=MAX_STEPS,
    save_steps=100,
    logging_steps=5,
    output_dir='outputs/checkpoints',
    report_to='none',
    log_completions=True,
)

# Standard TRL GRPO — reward_funcs receives completions + dataset columns as kwargs
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=compute_reward,    # callable(completions, **dataset_cols) -> list[float]
    args=training_args,
    train_dataset=dataset,
    callbacks=[RewardLogCallback()],
)

print(f'Starting GRPO training: {MAX_STEPS} steps | {len(dataset)} problems in dataset')
print(f'Grader: {SPACE_URL}/grader')
trainer.train()

## Cell 7 — Plot reward curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if reward_log:
    steps   = [r['step'] for r in reward_log]
    rewards = [r['reward'] for r in reward_log]

    # Smooth with moving average
    window  = max(2, len(rewards) // 10)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

    fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0F1117')
    ax.set_facecolor('#1A1D27')
    ax.bar(steps, rewards, color='#4C9BE8', alpha=0.4, width=0.8, label='Raw reward')
    ax.plot(steps[window-1:], smoothed, color='#FFD700', linewidth=2.5, label='Smoothed')
    ax.set_title('GRPO Training — Reward Curve', color='white', fontsize=14, fontweight='bold')
    ax.set_xlabel('Training Step', color='#AAAAAA')
    ax.set_ylabel('Reward', color='#AAAAAA')
    ax.tick_params(colors='#AAAAAA')
    ax.legend(facecolor='#1A1D27', labelcolor='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')
    plt.tight_layout()
    plt.savefig('grpo_reward_curve.png', dpi=150, facecolor='#0F1117')
    plt.show()
    print(f'Curve saved to grpo_reward_curve.png')
    print(f'Initial reward: {rewards[0]:.4f}')
    print(f'Final reward:   {rewards[-1]:.4f}')
    print(f'Improvement:    {rewards[-1] - rewards[0]:+.4f}')
else:
    print('No reward data yet — run Cell 6 first')

## Cell 8 — Save weights

In [ ]:
import os

# Save LoRA adapter — small, for resuming training (~20-50MB)
print('Saving LoRA adapter...')
model.save_pretrained('outputs/lora_adapter')
tokenizer.save_pretrained('outputs/lora_adapter')
print('LoRA adapter saved to outputs/lora_adapter')

# Merge LoRA into base model and save full model (~3GB)
print('Merging LoRA into base model...')
model.save_pretrained_merged(
    'outputs/merged_model',
    tokenizer,
    save_method='merged_16bit',
)
print('Merged model saved to outputs/merged_model')

## Cell 9 — Push to HuggingFace Hub (optional)

In [ ]:
# Only run this if you want to publish the trained model
# Set HF_REPO to your desired model name above

if HF_TOKEN and HF_REPO != 'YOUR_USERNAME/code-assessment-grpo':
    print(f'Pushing to https://huggingface.co/{HF_REPO}...')
    model.push_to_hub_merged(
        HF_REPO,
        tokenizer,
        save_method='merged_16bit',
        token=HF_TOKEN,
    )
    print(f'Model live at: https://huggingface.co/{HF_REPO}')
else:
    print('Skipping Hub push — set HF_TOKEN and HF_REPO in Cell 2 to enable')

## Cell 10 — Before vs After Comparison

Run the baseline (rule-based) agent for 20 episodes, then compare metrics
against the GRPO-trained model. This is the key evidence of training progress.

In [ ]:
import json, random, time
import matplotlib.pyplot as plt
import numpy as np

TASK_FORMATS = {
    'correctness_check':      ['correct, none','incorrect, factual-error','incorrect, instruction-violation','partially-correct, incomplete'],
    'tone_appropriateness':   ['appropriate, none','inappropriate, insensitive','inappropriate, too-technical','inappropriate, tone-mismatch'],
    'multi_dimensional':      ['correctness=8, tone=7, empathy=6, safety=9','correctness=2, tone=3, empathy=2, safety=4','correctness=5, tone=5, empathy=5, safety=5'],
    'conversation_coherence': ['consistent=yes, contradictions=0, context_loss=no','consistent=no, contradictions=2, context_loss=yes'],
    'adversarial_check':      ['issue=injection, severity=high','issue=none, severity=none','issue=format_violation, severity=low'],
}

def run_direct_agent(use_model=False, n_episodes=20):
    ep_rewards = []
    task_hits = {}; task_totals = {}
    for _ in range(n_episodes):
        env_inst = AIResponseEvalToolEnv()
        env_inst.reset()
        for _step in range(30):
            task = env_inst._obs.task_type if env_inst._obs else 'correctness_check'
            if use_model:
                ctx  = env_inst._obs.test_case_input if env_inst._obs else ''
                inp  = f'TASK: {task}\n{ctx}'
                toks = tokenizer(inp, return_tensors='pt').to(model.device)
                out  = model.generate(**toks, max_new_tokens=40, do_sample=False)
                answer = tokenizer.decode(out[0][toks.input_ids.shape[1]:], skip_special_tokens=True).strip()
            else:
                answer = random.choice(TASK_FORMATS.get(task, ['correct, none']))
            env_inst._obs = env_inst._env.step(AIResponseEvalAction(answer=answer))
            r = float(env_inst._obs.partial_credit or 0)
            env_inst.reward += r
            task_hits[task]   = task_hits.get(task, 0) + int(r >= 0.9)
            task_totals[task] = task_totals.get(task, 0) + 1
            if env_inst._obs.done: break
        ep_rewards.append(env_inst.reward / max(sum(task_totals.values()) or 1, 1))
    acc = {t: round(100*task_hits.get(t,0)/max(task_totals.get(t,1),1),1) for t in TASK_FORMATS}
    return ep_rewards, acc

print('Running baseline (20 episodes)...')
base_rew, base_acc = run_direct_agent(use_model=False)
print(f'Baseline avg reward: {sum(base_rew)/len(base_rew):.3f}')
print('Running trained model (20 episodes)...')
trained_rew, trained_acc = run_direct_agent(use_model=True)
print(f'Trained avg reward:  {sum(trained_rew)/len(trained_rew):.3f}')

BG,CARD,GOLD,BLUE,RED,GREY='#0F1117','#1A1D27','#FFD700','#4C9BE8','#E84C4C','#AAAAAA'
TASKS=['correctness_check','tone_appropriateness','multi_dimensional','conversation_coherence','adversarial_check']
LABELS=['Correctness','Tone','Multi-dim','Coherence','Adversarial']
fig,axes=plt.subplots(1,2,figsize=(14,5),facecolor=BG)
fig.suptitle('Before vs After GRPO Training',color='white',fontsize=14,fontweight='bold')
ax=axes[0]; ax.set_facecolor(CARD)
ax.plot(base_rew,color=RED,linewidth=2,label=f'Baseline (avg={sum(base_rew)/len(base_rew):.3f})')
ax.plot(trained_rew,color=GOLD,linewidth=2,label=f'GRPO (avg={sum(trained_rew)/len(trained_rew):.3f})')
ax.set_title('Avg Step Reward / Episode',color='white'); ax.set_xlabel('Episode',color=GREY)
ax.set_ylabel('Avg Reward',color=GREY); ax.tick_params(colors=GREY); ax.set_ylim(0,1)
ax.legend(facecolor=CARD,labelcolor='white')
for sp in ax.spines.values(): sp.set_edgecolor('#333355')
ax=axes[1]; ax.set_facecolor(CARD)
x=np.arange(len(LABELS)); w=0.35
ax.bar(x-w/2,[base_acc.get(t,0) for t in TASKS],w,color=RED,alpha=0.75,label='Before')
ax.bar(x+w/2,[trained_acc.get(t,0) for t in TASKS],w,color=BLUE,alpha=0.85,label='After')
ax.set_xticks(x); ax.set_xticklabels(LABELS,color=GREY,rotation=15,ha='right',fontsize=9)
ax.set_title('Per-task Accuracy',color='white'); ax.set_ylabel('Accuracy (%)',color=GREY)
ax.tick_params(colors=GREY); ax.set_ylim(0,100)
ax.legend(facecolor=CARD,labelcolor='white')
for sp in ax.spines.values(): sp.set_edgecolor('#333355')
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png',dpi=150,facecolor=BG,bbox_inches='tight')
plt.show()
print('Saved before_after_comparison.png')
print(f'\n{"Metric":<28} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-'*56)
print(f'{"Avg reward":<28} {sum(base_rew)/len(base_rew):>8.3f} {sum(trained_rew)/len(trained_rew):>8.3f} {sum(trained_rew)/len(trained_rew)-sum(base_rew)/len(base_rew):>+8.3f}')
for t,l in zip(TASKS,LABELS):
    b=base_acc.get(t,0); a=trained_acc.get(t,0)
    print(f'  {l:<26} {b:>7.1f}% {a:>7.1f}% {a-b:>+8.1f}%')


## Summary

| Output | Location | Description |
|---|---|---|
| LoRA adapter | `outputs/lora_adapter/` | Trained delta weights |
| Merged model | `outputs/merged_model/` | Full model for deployment |
| GRPO reward curve | `grpo_reward_curve.png` | Training progress |
| Before/After chart | `/content/before_after_comparison.png` | Real comparison |

**Reward going up in Cell 6 + accuracy jump in Cell 10 = proof of learning.**
